In [ ]:
import pdfplumber
import pandas as pd
import re
import os


pasta_pdfs = "C:/Users/Z0058B3H/Siemens Energy/Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly/1 - Relatório para extratificação de dados/VP1"
caminho_excel = "C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            
        if not texto_limpo:
            return dados_extraidos, f"{nome_arquivo} -> Sem texto."

       
        busca_inicial = re.search(padrao_inicial, texto_limpo)
        busca_final = re.search(padrao_final, texto_limpo)
        data_ini = busca_inicial.group(1) if busca_inicial else ""
        data_fin = busca_final.group(1) if busca_final else ""

    
        busca_duracao = re.search(padrao_duracao, texto_limpo)
        duracao = busca_duracao.group(1) if busca_duracao else ""

       
        matches = list(re.finditer(padrao_linha, texto_limpo))
        
        if not matches:
            return dados_extraidos, f"{nome_arquivo} -> Não encontrou o padrão no texto."



        massa_anterior = "Verificar no PDF"


        for match in matches:
            pa = match.group(1)
            texto_meio = match.group(2).strip()
            potencia = match.group(3)
            tensao = match.group(4)
            nr_abaix = match.group(5)
            mat_isol = match.group(6)
            
      
            if match.group(7):
                massa_total = match.group(7)
                massa_anterior = massa_total 
            else:
                massa_total = massa_anterior 
           
            partes_texto = texto_meio.split()
            if len(partes_texto) > 1:
                carregamento = partes_texto[-1]
                cliente = " ".join(partes_texto[:-1])
            else:
                carregamento = texto_meio
                cliente = "Não Identificado"

            dados_extraidos.append([
                pa, cliente, carregamento, potencia, tensao, 
                nr_abaix, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo
            ])
            
        return dados_extraidos, None

    except Exception as e:
        return dados_extraidos, f"{nome_arquivo} -> Erro inesperado: {e}"

lista_de_dados_geral = []
arquivos_com_erro = []

for nome_arquivo in os.listdir(pasta_pdfs):
    if nome_arquivo.lower().endswith(".pdf"):
        caminho_completo_pdf = os.path.join(pasta_pdfs, nome_arquivo)
        
        dados_do_arquivo, erro = processar_pdf(caminho_completo_pdf, nome_arquivo)
        
        if dados_do_arquivo:
            lista_de_dados_geral.extend(dados_do_arquivo)
            
        if erro:
            arquivos_com_erro.append(erro)

colunas = [
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", 
    "TENS. [kV]", "Nr. Abaix.", "MAT. ISOL [kg]", "MASSA TOTAL [Ton]",
    "DATA INICIAL", "DATA FINAL", "DURAÇÃO [horas]", "NOME_DO_ARQUIVO"
]

if lista_de_dados_geral:
    tabela_consolidada = pd.DataFrame(lista_de_dados_geral, columns=colunas)
    tabela_consolidada.to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if arquivos_com_erro:
    print("\nArquivos com problemas:")
    for erro in arquivos_com_erro:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_de_dados_geral)}")
print(f"Arquivos com falha/ignorados: {len(arquivos_com_erro)}")
print("="*50)

In [ ]:
import pdfplumber
import pandas as pd
import re
import os

pasta_pdfs = r"C:\teste"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

# Títulos das colunas da tabela que vamos adicionar
titulos_colunas_tabela = [
    "Dia-hora", "Fase", "Setp (Evaporador)", "Saida (Evaporador)", 
    "Ret (Evaporador)", "Evap (Evaporador)", "Evap Vác. (mbar)", 
    "T1 (Paredes)", "T2 (Paredes)", "T3 (Paredes)", "T4 (Paredes)", 
    "Sup (Jugo)", "Inf (Jugo)", "Sup (Bobina)", "Inf (Bobina)", 
    "Pirani (Vácuo mbar)", "Baratron (Vácuo mbar)", "Extr. Umid. (g/h.ton)"
]

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            
            if not texto_limpo:
                return dados_extraidos, f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""

            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            
            if not matches:
                return dados_extraidos, f"{nome_arquivo} -> Não encontrou o padrão no texto."

            # 2. EXTRAÇÃO E FILTRO DA TABELA (Pegando a primeira linha da fase VF)
            todas_as_linhas_tabela = []
            for num_pagina, pagina in enumerate(pdf.pages):
                tabela_extraida = pagina.extract_table()
                if tabela_extraida:
                    if num_pagina == 0:
                        todas_as_linhas_tabela.extend(tabela_extraida)
                    else:
                        todas_as_linhas_tabela.extend(tabela_extraida[2:]) 
            
            # Lista padrão vazia caso não encontre a tabela ou a fase VF
            primeira_linha_vf = [""] * len(titulos_colunas_tabela)

            if todas_as_linhas_tabela:
                df_tabela = pd.DataFrame(todas_as_linhas_tabela).fillna("")
                
                if len(df_tabela.columns) > 1:
                    # Filtra a fase "VF" e remove as últimas 7 colunas
                    df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
                    colunas_para_remover = df_tabela.columns[-7:]
                    df_tabela = df_tabela.drop(columns=colunas_para_remover)
                else:
                    # Lógica para quando o pdfplumber junta tudo em uma coluna
                    df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
                    df_tabela[0] = df_tabela[0].str.replace(r'(\s+[\d,]+){6}\s+\S+$', '', regex=True)
                    df_tabela = df_tabela[0].str.split(expand=True)

                # Se sobrou alguma linha após o filtro, pegamos a primeira
                if not df_tabela.empty:
                    primeira_linha_vf = df_tabela.iloc[0].tolist()
                    
                    # Garantir que a lista tem exatos 18 elementos para não quebrar o DataFrame depois
                    if len(primeira_linha_vf) < 18:
                        primeira_linha_vf.extend([""] * (18 - len(primeira_linha_vf)))
                    elif len(primeira_linha_vf) > 18:
                        primeira_linha_vf = primeira_linha_vf[:18]

            # 3. UNINDO OS DADOS DO CABEÇALHO COM A PRIMEIRA LINHA DA TABELA
            massa_anterior = "Verificar no PDF"

            for match in matches:
                pa = match.group(1)
                texto_meio = match.group(2).strip()
                potencia = match.group(3)
                tensao = match.group(4)
                nr_abaix = match.group(5)
                mat_isol = match.group(6)
                
                if match.group(7):
                    massa_total = match.group(7)
                    massa_anterior = massa_total 
                else:
                    massa_total = massa_anterior 
               
                partes_texto = texto_meio.split()
                if len(partes_texto) > 1:
                    carregamento = partes_texto[-1]
                    cliente = " ".join(partes_texto[:-1])
                else:
                    carregamento = texto_meio
                    cliente = "Não Identificado"

                # Cria a linha mesclando as informações (Cabeçalho + Tabela)
                linha_completa = [
                    pa, cliente, carregamento, potencia, tensao, 
                    nr_abaix, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo
                ] + primeira_linha_vf

                dados_extraidos.append(linha_completa)
                
        return dados_extraidos, None

    except Exception as e:
        return dados_extraidos, f"{nome_arquivo} -> Erro inesperado: {e}"

# Execução Principal
lista_de_dados_geral = []
arquivos_com_erro = []

for nome_arquivo in os.listdir(pasta_pdfs):
    if nome_arquivo.lower().endswith(".pdf"):
        caminho_completo_pdf = os.path.join(pasta_pdfs, nome_arquivo)
        
        dados_do_arquivo, erro = processar_pdf(caminho_completo_pdf, nome_arquivo)
        
        if dados_do_arquivo:
            lista_de_dados_geral.extend(dados_do_arquivo)
            
        if erro:
            arquivos_com_erro.append(erro)

# Definindo as colunas totais (Cabeçalho + Tabela)
colunas_cabecalho = [
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", 
    "TENS. [kV]", "Nr. Abaix.", "MAT. ISOL [kg]", "MASSA TOTAL [Ton]",
    "DATA INICIAL", "DATA FINAL", "DURAÇÃO [horas]", "NOME_DO_ARQUIVO"
]
colunas_totais = colunas_cabecalho + titulos_colunas_tabela

if lista_de_dados_geral:
    tabela_consolidada = pd.DataFrame(lista_de_dados_geral, columns=colunas_totais)
    tabela_consolidada.to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if arquivos_com_erro:
    print("\nArquivos com problemas:")
    for erro in arquivos_com_erro:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_de_dados_geral)}")
print(f"Arquivos com falha/ignorados: {len(arquivos_com_erro)}")
print("="*50)

In [ ]:
import pdfplumber
import pandas as pd
import re
import os
from tqdm import tqdm

pasta_pdfs = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

# Padrões Regex do Cabeçalho
padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

# Títulos EXATOS das colunas que você quer extrair da tabela
colunas_desejadas_tabela = [
    "Fase", "Sup (Jugo)", "Inf (Jugo)", "Sup (Bobina)", 
    "Inf (Bobina)", "Baratron (Vácuo mbar)", "Extr. Umid. (g/h.ton)"
]

# Índices dessas colunas dentro das 18 colunas originais do seu PDF
# 1=Fase, 11=Sup(Jugo), 12=Inf(Jugo), 13=Sup(Bobina), 14=Inf(Bobina), 16=Baratron, 17=Extr.Umid
indices_desejados = [1, 11, 12, 13, 14, 16, 17]

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            
            if not texto_limpo:
                return dados_extraidos, f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""

            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            
            if not matches:
                return dados_extraidos, f"{nome_arquivo} -> Não encontrou o padrão no texto."

            # 2. EXTRAÇÃO E FILTRO DA TABELA
            todas_as_linhas_tabela = []
            for num_pagina, pagina in enumerate(pdf.pages):
                tabela_extraida = pagina.extract_table()
                if tabela_extraida:
                    if num_pagina == 0:
                        todas_as_linhas_tabela.extend(tabela_extraida)
                    else:
                        todas_as_linhas_tabela.extend(tabela_extraida[2:]) 
            
            # Lista padrão vazia com o tamanho das 7 colunas desejadas (caso dê erro na tabela)
            primeira_linha_filtrada = [""] * len(colunas_desejadas_tabela)

            if todas_as_linhas_tabela:
                df_tabela = pd.DataFrame(todas_as_linhas_tabela).fillna("")
                
                if len(df_tabela.columns) > 1:
                    df_tabela = df_tabela[df_tabela[1].astype(str).str.strip() == "VF"]
                    colunas_para_remover = df_tabela.columns[-7:]
                    df_tabela = df_tabela.drop(columns=colunas_para_remover)
                else:
                    df_tabela = df_tabela[df_tabela[0].astype(str).str.contains(r'\bVF\b', regex=True)]
                    df_tabela[0] = df_tabela[0].str.replace(r'(\s+[\d,]+){6}\s+\S+$', '', regex=True)
                    df_tabela = df_tabela[0].str.split(expand=True)

                if not df_tabela.empty:
                    primeira_linha_completa = df_tabela.iloc[0].tolist()
                    
                    # Garante que a lista tenha pelo menos 18 itens para não dar erro de índice
                    if len(primeira_linha_completa) >= 18:
                        # Pega APENAS os índices que você quer (Fase, Jugo, Bobina, etc.)
                        primeira_linha_filtrada = [primeira_linha_completa[i] for i in indices_desejados]

            # 3. UNINDO OS DADOS
            massa_anterior = "Verificar no PDF"

            for match in matches:
                pa = match.group(1)
                texto_meio = match.group(2).strip()
                potencia = match.group(3)
                tensao = match.group(4)
                nr_abaix = match.group(5)
                mat_isol = match.group(6)
                
                if match.group(7):
                    massa_total = match.group(7)
                    massa_anterior = massa_total 
                else:
                    massa_total = massa_anterior 
               
                partes_texto = texto_meio.split()
                if len(partes_texto) > 1:
                    carregamento = partes_texto[-1]
                    cliente = " ".join(partes_texto[:-1])
                else:
                    carregamento = texto_meio
                    cliente = "Não Identificado"

                # Cria a linha mesclando (Cabeçalho + As 7 colunas da Tabela)
                linha_completa = [
                    pa, cliente, carregamento, potencia, tensao, 
                    nr_abaix, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo
                ] + primeira_linha_filtrada

                dados_extraidos.append(linha_completa)
                
        return dados_extraidos, None

    except Exception as e:
        return dados_extraidos, f"{nome_arquivo} -> Erro inesperado: {e}"

# ==============================================================================
# EXECUÇÃO PRINCIPAL
# ==============================================================================
lista_de_dados_geral = []
arquivos_com_erro = []

arquivos_pdf = [f for f in os.listdir(pasta_pdfs) if f.lower().endswith(".pdf")]

print("\nIniciando a extração dos dados...\n")

for nome_arquivo in tqdm(arquivos_pdf, desc="Processando PDFs", unit="arquivo"):
    caminho_completo_pdf = os.path.join(pasta_pdfs, nome_arquivo)
    
    dados_do_arquivo, erro = processar_pdf(caminho_completo_pdf, nome_arquivo)
    
    if dados_do_arquivo:
        lista_de_dados_geral.extend(dados_do_arquivo)
        
    if erro:
        arquivos_com_erro.append(erro)

# Definindo as colunas totais (Cabeçalho + As 7 da tabela filtrada)
colunas_cabecalho = [
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", 
    "TENS. [kV]", "Nr. Abaix.", "MAT. ISOL [kg]", "MASSA TOTAL [Ton]",
    "DATA INICIAL", "DATA FINAL", "DURAÇÃO [horas]", "NOME_DO_ARQUIVO"
]
colunas_totais = colunas_cabecalho + colunas_desejadas_tabela

if lista_de_dados_geral:
    tabela_consolidada = pd.DataFrame(lista_de_dados_geral, columns=colunas_totais)
    tabela_consolidada.to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if arquivos_com_erro:
    print("\nArquivos com problemas:")
    for erro in arquivos_com_erro:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_de_dados_geral)}")
print(f"Arquivos com falha/ignorados: {len(arquivos_com_erro)}")
print("="*50)

In [ ]:
import pdfplumber
import pandas as pd
import re
import os
from tqdm import tqdm

pasta_pdfs = r"C:\teste"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

# Padrões Regex do Cabeçalho
padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

def limpar_valor(valor):
    """Converte string do PDF (com vírgula) para float."""
    if valor and str(valor).strip():
        try:
            return float(str(valor).replace(',', '.'))
        except ValueError:
            return None
    return None

def calcular_media(val1, val2):
    v1 = limpar_valor(val1)
    v2 = limpar_valor(val2)
    valores = [v for v in (v1, v2) if v is not None]
    if valores:
        return round(sum(valores) / len(valores), 1)
    return 0.0

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            if not texto_limpo: return [], f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""
            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            if not matches: return [], f"{nome_arquivo} -> Não encontrou o padrão no texto."

            m = matches[0]
            pa, texto_meio, pot, tensao, nr_ab, mat_isol = m.group(1), m.group(2).strip(), m.group(3),

In [ ]:
import pdfplumber
import pandas as pd
import re
import os
from tqdm import tqdm

pasta_pdfs = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

# Padrões Regex do Cabeçalho
padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

def limpar_valor(valor):
    """Converte string do PDF (com vírgula) para float."""
    if valor and str(valor).strip():
        try:
            return float(str(valor).replace(',', '.'))
        except ValueError:
            return None
    return None

def calcular_media(val1, val2):
    v1 = limpar_valor(val1)
    v2 = limpar_valor(val2)
    valores = [v for v in (v1, v2) if v is not None]
    if valores:
        return round(sum(valores) / len(valores), 1)
    return 0.0

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            if not texto_limpo: return [], f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""
            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            if not matches: return [], f"{nome_arquivo} -> Não encontrou o padrão no texto."

            m = matches[0]
            pa, texto_meio, pot, tensao, nr_ab, mat_isol = m.group(1), m.group(2).strip(), m.group(3), m.group(4), m.group(5), m.group(6)
            massa_total = m.group(7) if m.group(7) else "Verificar"
            
            partes_texto = texto_meio.split()
            carregamento = partes_texto[-1] if len(partes_texto) > 1 else texto_meio
            cliente = " ".join(partes_texto[:-1]) if len(partes_texto) > 1 else "Não Identificado"

            cabecalho_base = [pa, cliente, carregamento, pot, tensao, nr_ab, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo]

            # 2. EXTRAÇÃO E PROCESSAMENTO DA TABELA
            todas_as_linhas = []
            for i, pag in enumerate(pdf.pages):
                tab = pag.extract_table()
                if tab: todas_as_linhas.extend(tab if i == 0 else tab[2:])

            if not todas_as_linhas:
                return [], f"{nome_arquivo} -> Tabela não encontrada."

            df_tab = pd.DataFrame(todas_as_linhas).fillna("")
            
            if len(df_tab.columns) > 1:
                df_vf = df_tab[df_tab[1].astype(str).str.strip() == "VF"].copy()
            else:
                df_vf = df_tab[df_tab[0].astype(str).str.contains(r'\bVF\b')].copy()
                df_vf = df_vf[0].str.split(expand=True)

            if df_vf.empty:
                return [], f"{nome_arquivo} -> Fase VF não encontrada."

            # Processar dados da tabela
            linhas_processadas = []
            for _, row in df_vf.iterrows():
                try:
                    dia_hora = str(row[0])
                    fase = str(row[1])
                    m_jugo = calcular_media(row[11], row[12])
                    m_bobina = calcular_media(row[13], row[14])
                    baratron = limpar_valor(row[16]) or 0.0
                    umid = limpar_valor(row[17]) or 0.0
                    
                    # Salva os valores originais em texto para colocar na tabela final
                    txt_baratron = str(row[16])
                    txt_umid = str(row[17])
                    
                    linhas_processadas.append({
                        'dados_planilha': [dia_hora, fase, m_jugo, m_bobina, txt_baratron, txt_umid],
                        'm_jugo': m_jugo, 'm_bobina': m_bobina, 'baratron': baratron, 'umid': umid
                    })
                except KeyError:
                    continue 

            # --- CONSTRUINDO A LINHA ÚNICA ---
            if linhas_processadas:
                # 1. Pega os dados da primeira aparição
                inicio_vf = linhas_processadas[0]['dados_planilha']
                
                # 2. Busca a linha que atende ao critério (se não achar, deixa em branco)
                final_vf = ["", "", "", "", "", ""] 
                for lp in linhas_processadas:
                    if (lp['m_bobina'] >= 100 and 
                        lp['m_jugo'] >= 90 and 
                        0 < lp['baratron'] <= 0.2 and 
                        lp['umid'] <= 5):
                        final_vf = lp['dados_planilha']
                        break
                
                # Junta TUDO na mesma linha (Cabeçalho + Início + Final)
                linha_completa = cabecalho_base + inicio_vf + final_vf
                dados_extraidos.append(linha_completa)

        return dados_extraidos, None
    except Exception as e:
        return [], f"{nome_arquivo} -> Erro: {e}"

# --- EXECUÇÃO PRINCIPAL ---
lista_geral = []
erros = []
arquivos = [f for f in os.listdir(pasta_pdfs) if f.lower().endswith(".pdf")]

print("\nIniciando a extração dos dados...\n")

for arq in tqdm(arquivos, desc="Processando PDFs", unit="arq"):
    res, erro = processar_pdf(os.path.join(pasta_pdfs, arq), arq)
    if res: lista_geral.extend(res)
    if erro: erros.append(erro)

# Colunas expandidas horizontalmente
colunas = [
    # Cabeçalho
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", "TENS. [kV]", "Nr. Abaix.", 
    "MAT. ISOL [kg]", "MASSA TOTAL [Ton]", "DATA INICIAL", "DATA FINAL", 
    "DURAÇÃO [horas]", "NOME_DO_ARQUIVO", 
    
    # Dados Início VF
    "Dia-Hora (Início)", "Fase (Início)", "Média Jugo (Início) [°C]", "Média Bobina (Início) [°C]", 
    "Baratron (Início) [mbar]", "Extr. Umid. (Início) [g/h.ton]",
    
    # Dados Critério Atingido VF
    "Dia-Hora (Final)", "Fase (Final)", "Média Jugo (Final) [°C]", "Média Bobina (Final) [°C]", 
    "Baratron (Final) [mbar]", "Extr. Umid. (Final) [g/h.ton]"
]

if lista_geral:
    pd.DataFrame(lista_geral, columns=colunas).to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if erros:
    print("\nArquivos com problemas:")
    for erro in erros:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_geral)}")
print(f"Arquivos com falha/ignorados: {len(erros)}")
print("="*50)

In [ ]:
#adiciona AER
import pdfplumber
import pandas as pd
import re
import os
from tqdm import tqdm

pasta_pdfs = r"C:\teste"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

# Padrões Regex do Cabeçalho
padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

def limpar_valor(valor):
    """Converte string do PDF (com vírgula) para float."""
    if valor and str(valor).strip():
        try:
            return float(str(valor).replace(',', '.'))
        except ValueError:
            return None
    return None

def calcular_media(val1, val2):
    v1 = limpar_valor(val1)
    v2 = limpar_valor(val2)
    valores = [v for v in (v1, v2) if v is not None]
    if valores:
        return round(sum(valores) / len(valores), 1)
    return 0.0

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            if not texto_limpo: return [], f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""
            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            if not matches: return [], f"{nome_arquivo} -> Não encontrou o padrão no texto."

            m = matches[0]
            pa, texto_meio, pot, tensao, nr_ab, mat_isol = m.group(1), m.group(2).strip(), m.group(3), m.group(4), m.group(5), m.group(6)
            massa_total = m.group(7) if m.group(7) else "Verificar"
            
            partes_texto = texto_meio.split()
            carregamento = partes_texto[-1] if len(partes_texto) > 1 else texto_meio
            cliente = " ".join(partes_texto[:-1]) if len(partes_texto) > 1 else "Não Identificado"

            cabecalho_base = [pa, cliente, carregamento, pot, tensao, nr_ab, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo]

            # 2. EXTRAÇÃO E PROCESSAMENTO DA TABELA
            todas_as_linhas = []
            for i, pag in enumerate(pdf.pages):
                tab = pag.extract_table()
                if tab: todas_as_linhas.extend(tab if i == 0 else tab[2:])

            if not todas_as_linhas:
                return [], f"{nome_arquivo} -> Tabela não encontrada."

            df_tab = pd.DataFrame(todas_as_linhas).fillna("")
            
            # Tratamento da tabela para suportar tanto AER quanto VF
            if len(df_tab.columns) > 1:
                df_limpo = df_tab.copy()
                df_limpo[1] = df_limpo[1].astype(str).str.strip()
            else:
                df_limpo = df_tab[df_tab[0].astype(str).str.contains(r'\b(VF|AER)\b')].copy()
                df_limpo = df_limpo[0].str.split(expand=True)
                if not df_limpo.empty and len(df_limpo.columns) > 1:
                    df_limpo[1] = df_limpo[1].astype(str).str.strip()

            # Variáveis padrão vazias (caso não encontre alguma informação)
            inicio_aer = ["", "", "", "", "", ""]
            inicio_vf = ["", "", "", "", "", ""]
            final_vf = ["", "", "", "", "", ""]

            if not df_limpo.empty and len(df_limpo.columns) > 1:
                df_aer = df_limpo[df_limpo[1] == "AER"]
                df_vf = df_limpo[df_limpo[1] == "VF"]

                # --- PROCESSAR AER (Pegar apenas a primeira aparição) ---
                for _, row in df_aer.iterrows():
                    try:
                        dia_hora = str(row[0])
                        fase = str(row[1])
                        m_jugo = calcular_media(row[11], row[12])
                        m_bobina = calcular_media(row[13], row[14])
                        txt_baratron = str(row[16]) if len(row) > 16 else ""
                        txt_umid = str(row[17]) if len(row) > 17 else ""
                        
                        inicio_aer = [dia_hora, fase, m_jugo, m_bobina, txt_baratron, txt_umid]
                        break # Encerra o loop no primeiro que encontrar
                    except KeyError:
                        continue

                # --- PROCESSAR VF ---
                if df_vf.empty:
                    return [], f"{nome_arquivo} -> Fase VF não encontrada."

                linhas_processadas_vf = []
                for _, row in df_vf.iterrows():
                    try:
                        dia_hora = str(row[0])
                        fase = str(row[1])
                        m_jugo = calcular_media(row[11], row[12])
                        m_bobina = calcular_media(row[13], row[14])
                        baratron = limpar_valor(row[16]) or 0.0
                        umid = limpar_valor(row[17]) or 0.0
                        txt_baratron = str(row[16])
                        txt_umid = str(row[17])
                        
                        linhas_processadas_vf.append({
                            'dados_planilha': [dia_hora, fase, m_jugo, m_bobina, txt_baratron, txt_umid],
                            'm_jugo': m_jugo, 'm_bobina': m_bobina, 'baratron': baratron, 'umid': umid
                        })
                    except KeyError:
                        continue 

                if linhas_processadas_vf:
                    # 1. Pega os dados da primeira aparição do VF
                    inicio_vf = linhas_processadas_vf[0]['dados_planilha']
                    
                    # 2. Busca a linha que atende ao critério final do VF
                    for lp in linhas_processadas_vf:
                        if (lp['m_bobina'] >= 100 and 
                            lp['m_jugo'] >= 90 and 
                            0 < lp['baratron'] <= 0.2 and 
                            lp['umid'] <= 5):
                            final_vf = lp['dados_planilha']
                            break
            else:
                return [], f"{nome_arquivo} -> Nenhuma fase VF ou AER encontrada de forma válida."

            # Junta TUDO na mesma linha (Cabeçalho + AER + Início VF + Final VF)
            linha_completa = cabecalho_base + inicio_aer + inicio_vf + final_vf
            dados_extraidos.append(linha_completa)

        return dados_extraidos, None
    except Exception as e:
        return [], f"{nome_arquivo} -> Erro: {e}"

# --- EXECUÇÃO PRINCIPAL ---
lista_geral = []
erros = []
arquivos = [f for f in os.listdir(pasta_pdfs) if f.lower().endswith(".pdf")]

print("\nIniciando a extração dos dados...\n")

for arq in tqdm(arquivos, desc="Processando PDFs", unit="arq"):
    res, erro = processar_pdf(os.path.join(pasta_pdfs, arq), arq)
    if res: lista_geral.extend(res)
    if erro: erros.append(erro)

# Colunas expandidas horizontalmente
colunas = [
    # Cabeçalho
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", "TENS. [kV]", "Nr. Abaix.", 
    "MAT. ISOL [kg]", "MASSA TOTAL [Ton]", "DATA INICIAL", "DATA FINAL", 
    "DURAÇÃO [horas]", "NOME_DO_ARQUIVO", 
    
    # Dados Primeira Aparição AER
    "Dia-Hora (AER)", "Fase (AER)", "Média Jugo (AER) [°C]", "Média Bobina (AER) [°C]", 
    "Baratron (AER) [mbar]", "Extr. Umid. (AER) [g/h.ton]",

    # Dados Início VF
    "Dia-Hora (Início VF)", "Fase (Início VF)", "Média Jugo (Início) [°C]", "Média Bobina (Início) [°C]", 
    "Baratron (Início) [mbar]", "Extr. Umid. (Início) [g/h.ton]",
    
    # Dados Critério Atingido VF
    "Dia-Hora (Final VF)", "Fase (Final VF)", "Média Jugo (Final) [°C]", "Média Bobina (Final) [°C]", 
    "Baratron (Final) [mbar]", "Extr. Umid. (Final) [g/h.ton]"
]

if lista_geral:
    pd.DataFrame(lista_geral, columns=colunas).to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if erros:
    print("\nArquivos com problemas:")
    for erro in erros:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_geral)}")
print(f"Arquivos com falha/ignorados: {len(erros)}")
print("="*50)

In [ ]:
#organiza dados de maneira diferente 
import pdfplumber
import pandas as pd
import re
import os
from tqdm import tqdm

pasta_pdfs = r"C:\Users\Z0058B3H\Siemens Energy\Junior, Edson Roberto Caceres - Project - Vapor phase drying - Block Assembly\1 - Relatório para extratificação de dados\VP1"
caminho_excel = r"C:/Users/Z0058B3H/Downloads/Dados_Transformador.xlsx"

# Padrões Regex do Cabeçalho
padrao_linha = r"(\d{4,7}[/-]\d{1,2})\s+(.*?)\s+(\d+(?:,\d+)?)\s+(\d+[Aa]?)\s+(\d+)\s+(\d+(?:,\d+)?)(?:\s+(\d+(?:,\d+)?))?(?=\s|$|DATA|DURAÇÃO|ESTUFA|SECAGEM)"
padrao_inicial = r"DATA INICIAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_final = r"DATA FINAL:\s*(\d{2}/\d{2}/\d{2}\s*\d{2}:\d{2})"
padrao_duracao = r"DURAÇÃO\s*\[horas\]:\s*(\d+)"

def limpar_valor(valor):
    """Converte string do PDF (com vírgula) para float."""
    if valor and str(valor).strip():
        try:
            return float(str(valor).replace(',', '.'))
        except ValueError:
            return None
    return None

def calcular_media(val1, val2):
    v1 = limpar_valor(val1)
    v2 = limpar_valor(val2)
    valores = [v for v in (v1, v2) if v is not None]
    if valores:
        return round(sum(valores) / len(valores), 1)
    return 0.0

def processar_pdf(caminho_completo_pdf, nome_arquivo):
    dados_extraidos = []
    
    try:
        with pdfplumber.open(caminho_completo_pdf) as pdf:
            # 1. EXTRAÇÃO DE TEXTO DO CABEÇALHO
            primeira_pag = pdf.pages[0]
            texto_limpo = primeira_pag.extract_text()
            if not texto_limpo: return [], f"{nome_arquivo} -> Sem texto."

            busca_inicial = re.search(padrao_inicial, texto_limpo)
            busca_final = re.search(padrao_final, texto_limpo)
            data_ini = busca_inicial.group(1) if busca_inicial else ""
            data_fin = busca_final.group(1) if busca_final else ""
            busca_duracao = re.search(padrao_duracao, texto_limpo)
            duracao = busca_duracao.group(1) if busca_duracao else ""

            matches = list(re.finditer(padrao_linha, texto_limpo))
            if not matches: return [], f"{nome_arquivo} -> Não encontrou o padrão no texto."

            m = matches[0]
            pa, texto_meio, pot, tensao, nr_ab, mat_isol = m.group(1), m.group(2).strip(), m.group(3), m.group(4), m.group(5), m.group(6)
            massa_total = m.group(7) if m.group(7) else "Verificar"
            
            partes_texto = texto_meio.split()
            carregamento = partes_texto[-1] if len(partes_texto) > 1 else texto_meio
            cliente = " ".join(partes_texto[:-1]) if len(partes_texto) > 1 else "Não Identificado"

            cabecalho_base = [pa, cliente, carregamento, pot, tensao, nr_ab, mat_isol, massa_total, data_ini, data_fin, duracao, nome_arquivo]

            # 2. EXTRAÇÃO E PROCESSAMENTO DA TABELA
            todas_as_linhas = []
            for i, pag in enumerate(pdf.pages):
                tab = pag.extract_table()
                if tab: todas_as_linhas.extend(tab if i == 0 else tab[2:])

            if not todas_as_linhas:
                return [], f"{nome_arquivo} -> Tabela não encontrada."

            df_tab = pd.DataFrame(todas_as_linhas).fillna("")
            
            if len(df_tab.columns) > 1:
                df_limpo = df_tab.copy()
                df_limpo[1] = df_limpo[1].astype(str).str.strip()
            else:
                df_limpo = df_tab[df_tab[0].astype(str).str.contains(r'\b(VF|AER)\b')].copy()
                df_limpo = df_limpo[0].str.split(expand=True)
                if not df_limpo.empty and len(df_limpo.columns) > 1:
                    df_limpo[1] = df_limpo[1].astype(str).str.strip()

            # Dicionário para organizar as variáveis nas colunas exatas
            d = {
                'hr_1vf': "", 'hr_2vf': "", 'hr_aer': "",
                'jg_1vf': "", 'jg_2vf': "", 'jg_aer': "",
                'bb_1vf': "", 'bb_2vf': "", 'bb_aer': "",
                'br_1vf': "", 'br_2vf': "", 'br_aer': "",
                'um_1vf': "", 'um_2vf': "", 'um_aer': ""
            }

            if not df_limpo.empty and len(df_limpo.columns) > 1:
                df_aer = df_limpo[df_limpo[1] == "AER"]
                df_vf = df_limpo[df_limpo[1] == "VF"]

                # --- PROCESSAR AER (Primeira aparição) ---
                for _, row in df_aer.iterrows():
                    try:
                        d['hr_aer'] = str(row[0])
                        d['jg_aer'] = calcular_media(row[11], row[12])
                        d['bb_aer'] = calcular_media(row[13], row[14])
                        d['br_aer'] = str(row[16]) if len(row) > 16 else ""
                        d['um_aer'] = str(row[17]) if len(row) > 17 else ""
                        break 
                    except KeyError:
                        continue

                # --- PROCESSAR VF ---
                if df_vf.empty:
                    return [], f"{nome_arquivo} -> Fase VF não encontrada."

                linhas_processadas_vf = []
                for _, row in df_vf.iterrows():
                    try:
                        linhas_processadas_vf.append({
                            'horario': str(row[0]),
                            'm_jugo': calcular_media(row[11], row[12]),
                            'm_bobina': calcular_media(row[13], row[14]),
                            'bar_val': limpar_valor(row[16]) or 0.0,
                            'umid_val': limpar_valor(row[17]) or 0.0,
                            'bar_txt': str(row[16]),
                            'umid_txt': str(row[17])
                        })
                    except KeyError:
                        continue 

                if linhas_processadas_vf:
                    # 1º VF (Primeira aparição)
                    l_ini = linhas_processadas_vf[0]
                    d['hr_1vf'] = l_ini['horario']
                    d['jg_1vf'] = l_ini['m_jugo']
                    d['bb_1vf'] = l_ini['m_bobina']
                    d['br_1vf'] = l_ini['bar_txt']
                    d['um_1vf'] = l_ini['umid_txt']
                    
                    # 2º VF (Critério Atingido)
                    for lp in linhas_processadas_vf:
                        if (lp['m_bobina'] >= 90 and lp['m_jugo'] >= 90 and 0 < lp['bar_val'] <= 0.5 and lp['umid_val'] <= 5):
                            d['hr_2vf'] = lp['horario']
                            d['jg_2vf'] = lp['m_jugo']
                            d['bb_2vf'] = lp['m_bobina']
                            d['br_2vf'] = lp['bar_txt']
                            d['um_2vf'] = lp['umid_txt']
                            break
            else:
                return [], f"{nome_arquivo} -> Nenhuma fase VF ou AER encontrada de forma válida."

            # Construção da linha na NOVA ordem que você pediu
            linha_completa = cabecalho_base + [
                # Horários
                d['hr_1vf'], d['hr_2vf'], d['hr_aer'],
                
                # Temperaturas (Jugo)
                d['jg_1vf'], d['jg_2vf'], d['jg_aer'],
                
                # Temperaturas (Bobina)
                d['bb_1vf'], d['bb_2vf'], d['bb_aer'],
                
                # Baratrons
                d['br_1vf'], d['br_2vf'], d['br_aer'],
                
                # Umidades
                d['um_1vf'], d['um_2vf'], d['um_aer']
            ]
            
            dados_extraidos.append(linha_completa)

        return dados_extraidos, None
    except Exception as e:
        return [], f"{nome_arquivo} -> Erro: {e}"

# --- EXECUÇÃO PRINCIPAL ---
lista_geral = []
erros = []
arquivos = [f for f in os.listdir(pasta_pdfs) if f.lower().endswith(".pdf")]

print("\nIniciando a extração dos dados...\n")

for arq in tqdm(arquivos, desc="Processando PDFs", unit="arq"):
    res, erro = processar_pdf(os.path.join(pasta_pdfs, arq), arq)
    if res: lista_geral.extend(res)
    if erro: erros.append(erro)

# Títulos das colunas refletindo a nova ordem
colunas = [
    # Cabeçalho
    "P.A.", "CLIENTE", "CARREGAMENTO", "POT. [MVA]", "TENS. [kV]", "Nr. Abaix.", 
    "MAT. ISOL [kg]", "MASSA TOTAL [Ton]", "DATA INICIAL", "DATA FINAL", 
    "DURAÇÃO [horas]", "NOME_DO_ARQUIVO", 
    
    # Horários
    "Horário (1º VF)", "Horário (2º VF)", "Horário (AER)",
    
    # Temperaturas Jugo
    "Média Jugo (1º VF) [°C]", "Média Jugo (2º VF) [°C]", "Média Jugo (AER) [°C]",
    
    # Temperaturas Bobina
    "Média Bobina (1º VF) [°C]", "Média Bobina (2º VF) [°C]", "Média Bobina (AER) [°C]",
    
    # Baratrons
    "Baratron (1º VF) [mbar]", "Baratron (2º VF) [mbar]", "Baratron (AER) [mbar]",
    
    # Umidades
    "Extr. Umid. (1º VF) [g/h.ton]", "Extr. Umid. (2º VF) [g/h.ton]", "Extr. Umid. (AER) [g/h.ton]"
]

if lista_geral:
    pd.DataFrame(lista_geral, columns=colunas).to_excel(caminho_excel, index=False)
    print(f"\nProcesso concluído! Sucesso total.")
else:
    print("\nNenhum dado extraído.")

if erros:
    print("\nArquivos com problemas:")
    for erro in erros:
        print(erro)

print("\n" + "="*50)
print("RELATÓRIO DE LEITURA:")
print(f"Linhas extraídas com sucesso: {len(lista_geral)}")
print(f"Arquivos com falha/ignorados: {len(erros)}")
print("="*50)